# 🎙️ RVC Inference — Load from Google Drive

**No training. No uploads. Just point to your Drive files and convert.**

### ✅ Required Google Drive Layout
```
MyDrive/
└── RVC_Packages/
    ├── pth/
    │   └── YourModel.pth
    ├── index/
    │   └── YourModel.index
    └── audio/
        └── input_audio.wav
```

> ⚡ Make sure you are using a **GPU runtime**:  
> `Runtime → Change runtime type → T4 GPU`

---

In [ ]:
#@title **Step 1 — Mount Google Drive**

from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

# Create expected folders in Drive if they don't exist yet
for folder in ['pth', 'index', 'audio']:
    os.makedirs(f'/content/drive/MyDrive/RVC_Packages/{folder}', exist_ok=True)

print("✅ Google Drive mounted.")
print()
print("📂 Upload your files to these Drive paths:")
print("   .pth   → MyDrive/RVC_Packages/pth/YourModel.pth")
print("   .index → MyDrive/RVC_Packages/index/YourModel.index")
print("   audio  → MyDrive/RVC_Packages/audio/input_audio.wav")

In [ ]:
#@title **Step 2 — Download & Extract RVC**
#@markdown This downloads the RVC runtime from Hugging Face (~a few hundred MB). Run once per session.

import os, zipfile, subprocess, sys
from IPython.display import clear_output
from ipywidgets import Button

zip_path     = "/content/RVC.zip"
extract_path = "/content/RVC"
hf_url       = "https://huggingface.co/datasets/BahaaMahmoud88/testrvc/resolve/main/RVC.zip"

if os.path.isdir(extract_path):
    print("ℹ️  /content/RVC already exists — skipping download.")
else:
    print("⬇️  Downloading RVC runtime ...")
    subprocess.check_call(["wget", "-q", "--show-progress", "-O", zip_path, hf_url])

    if not os.path.exists(zip_path) or os.path.getsize(zip_path) < 5_000_000:
        raise FileNotFoundError("❌ RVC.zip download failed or file is too small.")

    print("📦 Extracting ...")
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    os.remove(zip_path)
    print(f"✅ RVC extracted to {extract_path}")

clear_output()
Button(description="\u2714 Done 👍", button_style="success")

In [ ]:
#@title **Step 3 — Install RVC Dependencies**
#@markdown Installs PyTorch (CUDA 12.1), torchcrepe, and RVC requirements. Takes ~5 min.

import subprocess, sys
from IPython.display import clear_output
from ipywidgets import Button

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "-q"] + list(args))

def uv(*args):
    subprocess.check_call(["uv", "pip", "install", "--system"] + list(args))

pip("install", "uv")

print("📦 Installing PyTorch (CUDA 12.1) ...")
uv("torch", "torchvision", "torchaudio",
   "--index-url", "https://download.pytorch.org/whl/cu121")

print("📦 Installing torchcrepe ...")
uv("git+https://github.com/maxrmorrison/torchcrepe.git")

print("📦 Installing RVC requirements ...")
subprocess.check_call(
    ["uv", "pip", "install", "--system", "-r", "requirements-py312.txt"],
    cwd="/content/RVC"
)

# ── fairseq patch: weights_only=False (PyTorch 2.x compatibility) ─────────────
print("🩹 Applying fairseq patch ...")
patch_code = '''
import fairseq, os
p = os.path.join(os.path.dirname(fairseq.__file__), "checkpoint_utils.py")
src = open(p).read()
old = 'state = torch.load(f, map_location=torch.device("cpu"))'
new = 'state = torch.load(f, map_location=torch.device("cpu"), weights_only=False)'
if old in src:
    open(p, "w").write(src.replace(old, new))
    print("Patch applied.")
else:
    print("Already patched — skipped.")
'''
subprocess.check_call([sys.executable, "-c", patch_code])

clear_output()
Button(description="\u2714 Done 👍", button_style="success")

In [ ]:
#@title **Step 4 — Load Model Files from Drive**
#@markdown Set your model name below. Files are copied from Drive to local storage for faster I/O.

import os, shutil
from ipywidgets import Button

# ── ✏️  Set your file names here ───────────────────────────────────────────────
pth_filename   = "YourModel.pth"    #@param {type:"string"}
index_filename = "YourModel.index"  #@param {type:"string"}

# ── Drive source paths ────────────────────────────────────────────────────────
drive_pth   = f"/content/drive/MyDrive/RVC_Packages/pth/{pth_filename}"
drive_index = f"/content/drive/MyDrive/RVC_Packages/index/{index_filename}"

# ── Local destination paths ───────────────────────────────────────────────────
local_weights_dir = "/content/RVC/assets/weights"
local_index_dir   = "/content/RVC/logs/model"

os.makedirs(local_weights_dir, exist_ok=True)
os.makedirs(local_index_dir,   exist_ok=True)

local_pth   = os.path.join(local_weights_dir, pth_filename)
local_index = os.path.join(local_index_dir,   index_filename)

# ── Validate & copy .pth ──────────────────────────────────────────────────────
print("🔍 Checking .pth file ...")
if not os.path.exists(drive_pth):
    raise FileNotFoundError(
        f"❌ .pth not found at: {drive_pth}\n"
        "👉 Make sure the file is in MyDrive/RVC_Packages/pth/"
    )
shutil.copy2(drive_pth, local_pth)
print(f"✅ .pth copied  → {local_pth}")

# ── Validate & copy .index ────────────────────────────────────────────────────
print("🔍 Checking .index file ...")
if not os.path.exists(drive_index):
    raise FileNotFoundError(
        f"❌ .index not found at: {drive_index}\n"
        "👉 Make sure the file is in MyDrive/RVC_Packages/index/"
    )
shutil.copy2(drive_index, local_index)
print(f"✅ .index copied → {local_index}")

# ── Store paths for later cells ───────────────────────────────────────────────
import builtins
builtins.RVC_MODEL_PATH  = local_pth
builtins.RVC_INDEX_PATH  = local_index
print()
print("📌 Paths saved for inference:")
print(f"   Model : {local_pth}")
print(f"   Index : {local_index}")

Button(description="\u2714 Done 👍", button_style="success")

In [ ]:
#@title **Step 5 — Load Input Audio from Drive**
#@markdown Set the audio filename below. It must be in `MyDrive/RVC_Packages/audio/`.

import os, shutil
from IPython.display import Audio, display

# ── ✏️  Set your audio file name here ─────────────────────────────────────────
audio_filename = "input_audio.wav"  #@param {type:"string"}

drive_audio = f"/content/drive/MyDrive/RVC_Packages/audio/{audio_filename}"
local_audio = "/content/sample_data/input_audio.wav"

os.makedirs('/content/sample_data', exist_ok=True)

if not os.path.exists(drive_audio):
    raise FileNotFoundError(
        f"❌ Audio not found at: {drive_audio}\n"
        "👉 Make sure the file is in MyDrive/RVC_Packages/audio/"
    )

shutil.copy2(drive_audio, local_audio)
print(f"✅ Audio loaded from Drive → {local_audio}")
print()

import builtins
builtins.RVC_INPUT_PATH = local_audio

display(Audio(local_audio))

In [ ]:
#@title **Step 6 — Run Voice Conversion**
#@markdown Adjust pitch if needed, then run.

import os, subprocess, sys
import IPython.display as ipd
import builtins

# ── Retrieve paths set in previous cells ──────────────────────────────────────
model_path  = builtins.RVC_MODEL_PATH
index_path  = builtins.RVC_INDEX_PATH
input_path  = builtins.RVC_INPUT_PATH
output_path = "/content/RVC/audios/output_audio.wav"

# ── ✏️  Pitch shift ────────────────────────────────────────────────────────────
#@markdown **Pitch (semitones):** Male→Male or Female→Female = 0 | Female→Male = -12 | Male→Female = +12
pitch = 0  #@param {type:"slider", min:-12, max:12, step:1}

# ── Advanced parameters (change if needed) ────────────────────────────────────
f0_method            = "rmvpe"
index_rate           = 0.75
filter_radius        = 3
resample_sr          = 0
volume_normalization = 0.0
consonant_protection = 0.5

# ── Validate paths ────────────────────────────────────────────────────────────
for label, path in [("Model (.pth)", model_path),
                    ("Index (.index)", index_path),
                    ("Input audio", input_path)]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"❌ {label} not found: {path}")
    print(f"✅ {label}: {path}")

# ── Set environment variables expected by infer_cli.py ───────────────────────
os.environ['weight_root'] = os.path.dirname(model_path)
os.environ['index_root']  = os.path.dirname(index_path)

os.makedirs('/content/RVC/audios', exist_ok=True)
if os.path.exists(output_path):
    os.remove(output_path)

# ── Build inference command ───────────────────────────────────────────────────
cmd = [
    sys.executable, "tools/cmd/infer_cli.py",
    "--f0up_key",      str(pitch),
    "--input_path",    input_path,
    "--index_path",    os.path.basename(index_path),
    "--f0method",      f0_method,
    "--opt_path",      output_path,
    "--model_name",    os.path.basename(model_path),
    "--index_rate",    str(index_rate),
    "--device",        "cuda:0",
    "--is_half",       "True",
    "--filter_radius", str(filter_radius),
    "--resample_sr",   str(resample_sr),
    "--rms_mix_rate",  str(volume_normalization),
    "--protect",       str(consonant_protection),
]

print()
print("🚀 Running inference ...")
result = subprocess.run(cmd, cwd="/content/RVC")

if result.returncode != 0:
    raise RuntimeError("❌ Inference failed — see output above for details.")

if not os.path.exists(output_path):
    raise FileNotFoundError("❌ Output audio was not created. Check the error log above.")

print()
print("✅ Inference complete! Previewing output ...")
print(f"   Saved at: {output_path}")
ipd.display(ipd.Audio(output_path))

In [ ]:
#@title **Step 7 — Save Output Audio to Google Drive**
#@markdown Copies `output_audio.wav` back to `MyDrive/RVC_Packages/audio/output_audio.wav`.

import os, shutil

src = "/content/RVC/audios/output_audio.wav"
dst = "/content/drive/MyDrive/RVC_Packages/audio/output_audio.wav"

if not os.path.exists(src):
    print(f"❌ Output not found: {src}\nRun Step 6 first.")
else:
    shutil.copy2(src, dst)
    print(f"✅ Output saved to Google Drive:")
    print(f"   {dst}")

In [ ]:
#@title **Optional — Download Output to Local Machine**
#@markdown Use this if you prefer a direct browser download instead of Drive.

from google.colab import files
import os

file_path = "/content/RVC/audios/output_audio.wav"

if os.path.exists(file_path):
    files.download(file_path)
else:
    print(f"❌ File not found: {file_path}\nRun Step 6 first.")